# 04 — Predictive Modeling with Spatially-Aware Validation

**Objective:** Build interpretable, generalizable models for response time prediction,
validated with spatial cross-validation to prevent data leakage.

**Methodology:**
- `GroupKFold` by province (not random split) to prevent spatial leakage
- Both overall and worst-province MAE reported for equity assessment
- Two model families compared: Random Forest and Histogram Gradient Boosting

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT_ROOT = Path('.')
V3_DIR = PROJECT_ROOT / 'mda_project' / 'data' / 'processed_v3'

mission = pd.read_parquet(V3_DIR / 'mission_records_v3.parquet').copy()
mission['t0'] = pd.to_datetime(mission['t0'], errors='coerce')
mission['hour'] = mission['t0'].dt.hour
mission['weekday'] = mission['t0'].dt.weekday
mission['month'] = mission['t0'].dt.month
mission['is_weekend'] = mission['weekday'].isin([5, 6]).astype(int)

feature_cols = ['latitude','longitude','dist_to_aed_km','hour','weekday','month',
                'is_weekend','province','event_type','event_level','n_dispatch','n_vector_types']
X = mission[feature_cols]
y = mission['response_min']
groups = mission['province'].fillna('UNKNOWN')

print(f'Training samples: {len(X):,} | Features: {len(feature_cols)}')

## 4.1 Model Training with GroupKFold Cross-Validation

In [ ]:
num_cols = ['latitude','longitude','dist_to_aed_km','hour','weekday','month',
            'is_weekend','n_dispatch','n_vector_types']
cat_cols = ['province','event_type','event_level']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])

models = {
    'Random Forest': RandomForestRegressor(n_estimators=350, random_state=42, n_jobs=-1, min_samples_leaf=2),
    'Hist. Gradient Boosting': HistGradientBoostingRegressor(random_state=42, max_depth=8, learning_rate=0.05),
}

gkf = GroupKFold(n_splits=5)
results = []

for name, model in models.items():
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups), start=1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]

        Xt_tr = preprocessor.fit_transform(Xtr)
        Xt_te = preprocessor.transform(Xte)
        model.fit(Xt_tr, ytr)
        pred = model.predict(Xt_te)

        province_te = groups.iloc[te]
        per_prov_mae = pd.DataFrame({'province': province_te.values, 'y': yte.values, 'pred': pred})
        prov_mae = per_prov_mae.groupby('province').apply(lambda d: mean_absolute_error(d['y'], d['pred']))

        results.append({
            'Model': name, 'Fold': fold,
            'MAE': mean_absolute_error(yte, pred),
            'RMSE': mean_squared_error(yte, pred, squared=False),
            'R2': r2_score(yte, pred),
            'Worst Province': prov_mae.idxmax(),
            'Worst Province MAE': float(prov_mae.max()),
        })

eval_df = pd.DataFrame(results)
print(eval_df)

## 4.2 Model Comparison Summary

In [ ]:
summary = eval_df.groupby('Model').agg(
    MAE_mean=('MAE','mean'), RMSE_mean=('RMSE','mean'),
    R2_mean=('R2','mean'), Worst_MAE_mean=('Worst Province MAE','mean')
).sort_values('MAE_mean')

print(summary.style.format('{:.3f}'))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for name in models:
    subset = eval_df[eval_df['Model'] == name]
    ax.plot(subset['Fold'], subset['MAE'], marker='o', linewidth=2, label=f'{name} (mean={subset["MAE"].mean():.2f})')

ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('MAE [min]', fontsize=12)
ax.set_title('Spatial Cross-Validation: MAE per Fold', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

**Modeling principles:**
- `GroupKFold(province)` prevents spatial leakage, producing conservative but honest estimates
- We report both overall error and worst-province error to assess spatial equity
- These tabular models serve as interpretable baselines; **Notebook 06** extends to deep spatiotemporal models